# 🎮 实验 8：Atari 上的深度 Q 网络（DQN）

在本实验中，你将把强化学习技能从经典控制环境（例如 **CartPole**）扩展到更复杂的 **Atari 游戏**，例如 *Pong-v5*。

你将实现一个 **深度 Q 网络（Deep Q-Network, DQN）**，使智能体能够直接从原始像素观测中学习。

### 学习目标

- 理解 DQN 如何将 **Q-learning** 与 **深度神经网络** 结合起来，以处理高维视觉输入。
- 实现 DQN 的核心组件：
  - 经验回放缓冲区（Replay Buffer）
  - 目标网络（Target Network）
  - ε-贪心探索策略（ε-Greedy Exploration Strategy）
- 训练一个智能体，使其能够在 Atari 环境中达到有意义的表现。
- 可视化训练过程，并展示录制的游戏画面帧。

### 第 1 部分：环境配置

在开始本实验之前，你需要创建一个新的 Conda 环境（Python 3.10），并安装 Atari 强化学习所需的软件包。

- 步骤 1：创建并激活环境
```bash
conda create -n atari python=3.10 -y
conda activate atari

- 步骤 2：使用 pip 安装支持 Atari 的 Gymnasium、PyTorch，以及本实验后续会用到的工具包。
```bash
pip install gymnasium[atari,accept-rom-license]==0.29.1
pip install autorom[accept-rom-license]
pip install stable-baselines3[extra]
pip install opencv-python imageio matplotlib
AutoROM --accept-license 

- 步骤 3：安装 Torch 和 TorchRL。
```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
pip install torchrl
pip install "gymnasium[atari,accept-rom-license]==0.29.1" "autorom[accept-rom-license]" ale-py
AutoROM --accept-license

pip install jupyter
pip install ipykernel

- 步骤 4：使用 “Pong” 环境验证安装是否成功。

In [15]:
import gymnasium as gym
import numpy as np

env = gym.make("ALE/Pong-v5", render_mode="rgb_array")
frames = []

obs, info = env.reset(seed=0)
done = False
while not done:
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    frames.append(env.render())  
    done = terminated or truncated

env.close()

In [16]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

#frames = np.load("pong_frames_uint8.npy")
N = len(frames)

out = widgets.Output()
slider = widgets.IntSlider(min=0, max=N-1, step=1, value=0, description="Frame")

@widgets.interact(i=slider)
def _show(i):
    with out:
        clear_output(wait=True)
        plt.imshow(frames[i])
        plt.axis('off')
        plt.show()

display(out)


interactive(children=(IntSlider(value=0, description='Frame', max=842), Output()), _dom_classes=('widget-inter…

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 640x480 with 1 Axes>', 'i…

- Atari 环境概述

**Atari 环境** 是强化学习研究中最常用的基准测试环境之一。  
这些环境提供了视觉信息丰富且具有挑战性的任务，使智能体能够直接从 **原始像素输入** 中学习控制策略。  
Atari 环境属于 **Arcade Learning Environment（ALE）**，可以通过 **Gymnasium** 调用。

- 环境组成部分

| 组成部分 | 描述 |
|------------|--------------|
| **状态（Observation）** | 游戏画面的原始 RGB 图像，尺寸为 **(210 × 160 × 3)**。对于 DQN，这些帧通常会被转换为灰度图，调整大小（例如 84 × 84），并进行帧堆叠（例如堆叠 4 帧），以提供时间上下文信息。 |
| **动作空间（Action Space）** | 一组离散的有效摇杆动作，不同游戏的动作空间可能不同。例如，在 **Pong** 中，有 6 个可能动作：<br> `0: NOOP`（无操作）<br> `1: FIRE`（开始游戏）<br> `2: MOVE RIGHT`（向右移动）<br> `3: MOVE LEFT`（向左移动）<br> `4: MOVE UP`（向上移动）<br> `5: MOVE DOWN`（向下移动） |
| **奖励（Reward）** | 每次执行动作后环境返回的标量反馈信号。<br> • 在 **Pong** 中，当智能体得分时奖励为 +1，当对手得分时奖励为 -1。<br> • 在 **Breakout** 中，智能体每击碎一块砖可获得 +1 奖励。<br> 累积奖励反映了智能体的游戏得分。 |
| **终止标志（Done Flag）** | 表示游戏是否已经结束，例如胜利、失败，或达到最大步数限制。 |

#### 常见预处理步骤

为了稳定学习过程，通常会对观测进行如下预处理：

1. 将 RGB 图像帧转换为灰度图。
2. 将图像尺寸调整为 84 × 84。
3. 堆叠最近的 4 帧图像。
4. 将像素值归一化到 `[0, 1]`。

这样可以降低计算成本，并帮助智能体感知运动信息。

### 第 2 部分：TorchRL 简介

TorchRL 是一个基于 PyTorch 的 **强化学习（Reinforcement Learning, RL）** 研究与教学库。它提供了一个模块化框架，将环境、数据收集、经验回放缓冲区、数据变换以及策略学习整合在一起，并且全部建立在 **PyTorch** 和 **TensorDict** 之上。

---

传统的强化学习实现通常需要大量样板代码，例如：

- 环境封装与预处理，例如灰度化、图像缩放、帧堆叠等
- 经验回放缓冲区的设计与采样
- 批量 rollout 与异步数据收集
- 与 PyTorch 张量和 GPU 设备之间的稳定接口

TorchRL 通过统一的数据结构和模块化组件简化了这些任务，使强化学习实验更加 **可复现** 且 **可扩展**。下面是 TorchRL 的核心概念。

---

| 概念 | 描述 |
|----------|-------------|
| **`TensorDict`** | 一种类似字典的容器，用于存储具有一致 batch 形状的张量，例如观测、动作、奖励等。 |
| **`EnvBase` / `GymEnv`** | TorchRL 中的环境基类，兼容 Gymnasium 环境，例如 `ALE/Pong-v5`。 |
| **`TransformedEnv`** | 一种环境封装器，可以自动对环境应用一系列 **变换（transforms）**，例如灰度化、尺寸调整、归一化等。 |
| **`ReplayBuffer`** | 用于存储和采样 transition 的记忆模块。TorchRL 支持简单经验回放缓冲区和优先级经验回放缓冲区。 |
| **`Collector`** | 用于高效处理 rollout 和数据收集，支持多个并行环境。 |
| **`LossModules`** | 已经实现好的损失函数模块，可用于 DQN、A2C、PPO、SAC 等算法。 |

---

In [17]:
import torch
from torchrl.envs import GymEnv, TransformedEnv, Compose
from torchrl.envs.transforms import ToTensorImage, GrayScale, Resize, CatFrames, DoubleToFloat, RewardClipping
from torchrl.data import TensorDictReplayBuffer, LazyMemmapStorage
from torchrl.data.replay_buffers.samplers import RandomSampler

首先，我们将通过从 Atari 环境中采样轨迹，介绍 **TensorDict** 和 **TorchRL 经验回放缓冲区**。

在 TorchRL 中，从环境中收集到的数据，例如观测、动作、奖励和下一状态，都会被存储在 **TensorDict** 中。  
**TensorDict** 是一种类似字典的容器，用于保存具有一致 batch 形状的 PyTorch 张量。

每一次环境交互都会返回一个 TensorDict。它以结构化且支持设备管理的格式组织数据，因此可以方便地进行数据操作、变换，并存储起来供后续训练使用。

原始的 Atari 图像帧是高维 RGB 图像，尺寸为 210×160×3，并不适合直接用于深度 Q-learning。  
TorchRL 支持使用 **环境变换（environment transforms）** 进行自动预处理，即通过 `TransformedEnv` 对基础环境进行封装。  
每个 transform 都会在 TensorDict 被返回之前，对其中的观测数据进行修改。

Atari 上 DQN 常用的典型变换包括：

- `ToTensorImage()`：将图像从 HWC 格式的 uint8 转换为 CHW 格式的 float，并将像素值缩放到 [0, 1]
- `GrayScale()`：将 RGB 图像转换为灰度图，将输入通道数从 3 降低到 1
- `Resize(84, 84)`：将图像帧调整为标准的 84×84 输入尺寸
- `CatFrames(N=4)`：堆叠最近的 4 帧图像，以捕捉运动信息
- `RewardClipping(-1, 1)`：通过限制奖励幅度来稳定训练过程
- `DoubleToFloat()`：确保网络输入使用 float32 精度

下面给出了一个使用 TorchRL 构建预处理 Atari 环境的示例：

In [4]:
# Base Gymnasium environment
base_env = GymEnv("ALE/Pong-v5", from_pixels=True, pixels_only=True, render_mode="rgb_array")
n_actions = base_env.action_space.n
obs_shape = (4, 84, 84)

# Apply preprocessing transforms
env = TransformedEnv(
    base_env,
    Compose(
        ToTensorImage(),         # Convert to tensor format
        GrayScale(),             # Convert RGB → grayscale
        Resize(84, 84),          # Resize to 84×84
        CatFrames(N=4, dim=-3),  # Stack 4 frames → (4, 84, 84)
        DoubleToFloat(),         # Ensure float32 precision
        RewardClipping(-1, 1),   # Clip rewards to [-1, 1]
    ),
)

#### TensorDict：核心数据容器

TensorDict 的使用方式类似字典，但它会确保其中保存的所有张量具有一致的 batch 维度。

例如，环境执行一步交互后，可能会产生如下形式的 TensorDict：

```python
TensorDict({
    'pixels': Tensor(...),          # 当前观测
    'action': Tensor(...),
    'next': {
        'pixels': Tensor(...),      # 下一时刻观测
        'reward': Tensor(...),
        'done': Tensor(...)
    }
}) 

#### TorchRL 原生经验回放缓冲区

TorchRL 提供了一套强大且灵活的经验回放缓冲区系统，其核心数据结构是 **TensorDict**。  
`TensorDictReplayBuffer` 类配合 `LazyMemmapStorage` 使用，可以高效地存储并采样从环境中收集到的 transition。

其主要优点包括：

- 与基于 TensorDict 的环境无缝集成
- 自动处理设备，例如 CPU/GPU
- 同时支持内存存储和磁盘后端存储
- 内置随机采样和优先级采样策略

In [5]:
from torchrl.data.replay_buffers.samplers import RandomSampler
from torchrl.data import TensorDictReplayBuffer, LazyMemmapStorage
from tensordict import TensorDict

rb = TensorDictReplayBuffer(
    storage=LazyMemmapStorage(max_size=50_000),  # disk-backed storage (efficient and scalable)
    sampler=RandomSampler(),                     # uniform random sampling
    batch_size=32,                               # default sample batch size
)

In [6]:
# Collect a trajectory with random transitions
td = env.reset()
for _ in range(5000):
    
    # sample an action
    a = env.action_spec.rand()
    obs = td.get("pixels")
    td = env.step(td.set("action", a))
    next_obs = td.get(("next", "pixels"))
    r = td.get(("next", "reward"))
    d = td.get(("next", "done"))

    transition = TensorDict(
        {
            "obs": obs,
            "action": a,
            "reward": r,
            "next_obs": next_obs,
            "done": d,
        },
        batch_size=[],
    )
    rb.add(transition)

    if d.item():
        td = env.reset()

print("Replay buffer size:", len(rb))

Replay buffer size: 5000


🔍 说明：`td` 和 `transition` 的区别

| 变量 | 作用 | 结构 | 用途 |
|-----------|------|------------|--------|
| **`td`** | 环境通过 `env.reset()` 或 `env.step()` 返回的实时 **TensorDict**。它同时包含当前步和下一步的信息，其中下一步信息嵌套在 `"next"` 字段下。 | `{ "action": ..., "next": { "pixels": ..., "reward": ..., "done": ... } }` | 用于 **与环境交互**，例如传入 `env.step()`，并在每次执行动作后更新。 |
| **`transition`** | 从 `td` 中相关字段整理得到的 **扁平化 TensorDict**，包含一个完整的经验样本 \((s_t, a_t, r_t, s_{t+1}, done_t)\)。 | `{ "obs": ..., "action": ..., "reward": ..., "next_obs": ..., "done": ... }` | 用于 **存入经验回放缓冲区**，之后再采样出来用于训练，例如 DQN 的参数更新。 |

**简单来说：**

- `td` 是环境在当前交互步骤中返回的 **结构化实时输出**。
- `transition` 是一个经验 transition 的 **简化快照**，会被存入 replay buffer。

In [7]:
batch = rb.sample(32)

### 第 3 部分：Atari 上的 DQN

在本部分中，你将完成 **深度 Q 网络（Deep Q-Network, DQN）** 算法在 Atari 环境中的实现，例如 *Pong*。  
提供的代码已经初始化了环境、经验回放缓冲区和 Q 网络。你的任务是将这些组件 **连接起来**，形成完整的 DQN 学习流程。

---

### ⚙️ 背景知识

DQN 通过最小化 **Bellman 误差（Bellman error）**，学习一个近似的动作价值函数 $ Q_\theta(s, a) $：

$$
L(\theta) = \mathbb{E}\Big[(Q_\theta(s_t, a_t) - y_t)^2\Big],
$$

其中

$$
y_t = r_t + \gamma (1 - d_t) \max_{a'} Q_{\theta^-}(s_{t+1}, a')
$$

并且 $ Q_{\theta^-} $ 是使用延迟参数的 **目标网络（target network）**。

---

### 🧩 已提供的组件

你已经获得了：

- ✅ 一个 TorchRL Atari 环境（`env`）
- ✅ 经验回放缓冲区 `rb`
- ✅ 在线 Q 网络和目标 Q 网络（`q`, `q_target`）
- ✅ 优化器和折扣因子（`optimizer`, `gamma`）
- ✅ 已经写好的代码，包括：
  - 从 `rb` 中采样
  - 计算 `target` 和 `loss`
  - 执行一次梯度更新

接下来，你将编写 **训练循环（training loop）**，把这些部分组合起来。

---

In [8]:
# Your time to work on it (See below for some hints)

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [ ]:
'''
提示 1：神经网络设计
'''
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class QNet(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512), nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        # x: (B,4,84,84) float32 in [0,1]
        z = self.conv(x)
        z = z.view(z.size(0), -1)
        return self.fc(z)

q = QNet(n_actions).to(device)
q_target = QNet(n_actions).to(device)
q_target.load_state_dict(q.state_dict())
q_target.eval()

optimizer = optim.Adam(q.parameters(), lr=1e-4)
gamma = 0.99

In [ ]:
'''
提示 2：如何为 Q-learning 实现梯度下降
'''
batch = rb.sample(batch_size)
obs_b      = batch["obs"].to(device)              # (B,4,84,84)
act_b      = batch["action"].long().to(device)    # (B,)
rew_b      = batch["reward"].to(device).squeeze(-1)  # make sure it's (B,)
next_obs_b = batch["next_obs"].to(device)         # (B,4,84,84)
done_b     = batch["done"].to(device).float().squeeze(-1)


with torch.no_grad():
    # target = r + gamma * (1-done) * max_a' Q_target(s',a')
    q_next = q_target(next_obs_b).max(1).values
    target = rew_b + gamma * (1.0 - done_b) * q_next

act_b_ind = act_b.argmax(dim=-1)
q_values = q(obs_b).gather(1, act_b_ind.view(-1, 1)).squeeze(1)
loss = F.smooth_l1_loss(q_values, target)

optimizer.zero_grad(set_to_none=True)
loss.backward()
nn.utils.clip_grad_norm_(q.parameters(), max_norm=10.0)
optimizer.step()

NameError: name 'batch_size' is not defined

In [ ]:
# 提示 3: Epsilon-greedy policy
def select_action(obs, eps: float):
    if torch.rand(1).item() < eps:
        # Use TorchRL action_spec for a proper tensor action
        return env.action_spec.rand()  # scalar tensor (long)
    with torch.no_grad():
        x = obs.unsqueeze(0).to(device)   # (1,4,84,84)
        qvals = q(x)                               # (1,n_actions)
        a = torch.argmax(qvals, dim=1).to("cpu")   # back to CPU
        return a.squeeze(0)      